In [1]:
!pip install cryptography

In [2]:
import sqlite3
import hashlib
import secrets
import base64

from cryptography.hazmat.primitives.ciphers.aead import AESGCM

In [3]:
connection = sqlite3.connect("secure_user_database.db")

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password_hash TEXT NOT NULL,
    encrypted_info TEXT NOT NULL
)
""")

connection.commit()

print("Secure database created successfully.")

Secure database created successfully.


In [4]:
aes_key = AESGCM.generate_key(bit_length=256)

print("AES-256 encryption key generated successfully.")
print("Key length:", len(aes_key) * 8, "bits")

AES-256 encryption key generated successfully.
Key length: 256 bits


In [5]:
def encrypt_data(data):
    aesgcm = AESGCM(aes_key)
    
    nonce = secrets.token_bytes(12)
    encrypted = aesgcm.encrypt(
        nonce,
        data.encode(),
        None
    )
    
    return base64.b64encode(nonce + encrypted).decode()

In [6]:
def decrypt_data(encrypted_data):
    aesgcm = AESGCM(aes_key)
    
    raw_data = base64.b64decode(encrypted_data)
    
    nonce = raw_data[:12]
    encrypted = raw_data[12:]
    
    decrypted = aesgcm.decrypt(
        nonce,
        encrypted,
        None
    )
    
    return decrypted.decode()

In [7]:
def hash_password(password):
    return hashlib.sha256(
        password.encode()
    ).hexdigest()

In [8]:
username = "admin"
password = "Admin@123"
sensitive_info = "Confidential User Information"

password_hash = hash_password(password)
encrypted_info = encrypt_data(sensitive_info)

cursor.execute("""
INSERT OR IGNORE INTO users
(username, password_hash, encrypted_info)
VALUES (?, ?, ?)
""", (
    username,
    password_hash,
    encrypted_info
))

connection.commit()

print("User added securely.")

User added securely.


In [9]:
cursor.execute("SELECT * FROM users")

users = cursor.fetchall()

for user in users:
    print(user)

(1, 'admin', 'e86f78a8a3caf0b60d8e74e5942aa6d86dc150cd3c03338aef25b7d2d7e3acc7', 'C9RO4g2x/Q5ugpOjACy0OXSvI1gaoMF/q+glDM0aHLzz4PYfDfMBg27rY4zy+8Y6eoQEHUKRsZTj')


In [10]:
def secure_login(username, password):
    
    password_hash = hash_password(password)
    
    query = """
    SELECT username, encrypted_info
    FROM users
    WHERE username = ?
    AND password_hash = ?
    """
    
    cursor.execute(query, (username, password_hash))
    
    result = cursor.fetchone()
    
    if result:
        print("Login successful.")
        print("Encrypted information:", result[1])
        print("Decrypted information:", decrypt_data(result[1]))
        return True
    
    else:
        print("Login failed.")
        return False

In [11]:
secure_login("admin", "Admin@123")

Login successful.
Encrypted information: C9RO4g2x/Q5ugpOjACy0OXSvI1gaoMF/q+glDM0aHLzz4PYfDfMBg27rY4zy+8Y6eoQEHUKRsZTj
Decrypted information: Confidential User Information


True

In [12]:
secure_login("admin", "WrongPassword")

Login failed.


False

In [13]:
suspicious_username = "' OR '1'='1"

print("Testing suspicious input:")
print(suspicious_username)

secure_login(suspicious_username, "anything")

Testing suspicious input:
' OR '1'='1
Login failed.


False

In [14]:
capability_code = secrets.token_urlsafe(16)

print("Capability code generated.")
print("Code:", capability_code)

Capability code generated.
Code: WBcaUESfy6F108OKB8F0RQ


In [15]:
def verify_capability_code(user_code):
    
    if secrets.compare_digest(
        user_code,
        capability_code
    ):
        print("Capability verification successful.")
        return True
    
    else:
        print("Capability verification failed.")
        return False

In [16]:
verify_capability_code(capability_code)

Capability verification successful.


True

In [17]:
verify_capability_code("wrong-code")

Capability verification failed.


False

In [18]:
def secure_access(username, password, user_code):
    
    print("Layer 1: Checking capability code...")
    
    if not verify_capability_code(user_code):
        print("Access denied.")
        return
    
    print("Layer 2: Checking secure login...")
    
    if secure_login(username, password):
        print("Secure access granted.")
    else:
        print("Access denied.")

In [19]:
secure_access(
    "admin",
    "Admin@123",
    capability_code
)

Layer 1: Checking capability code...
Capability verification successful.
Layer 2: Checking secure login...
Login successful.
Encrypted information: C9RO4g2x/Q5ugpOjACy0OXSvI1gaoMF/q+glDM0aHLzz4PYfDfMBg27rY4zy+8Y6eoQEHUKRsZTj
Decrypted information: Confidential User Information
Secure access granted.


In [20]:
secure_access(
    "' OR '1'='1",
    "anything",
    capability_code
)

Layer 1: Checking capability code...
Capability verification successful.
Layer 2: Checking secure login...
Login failed.
Access denied.


In [21]:
secure_access(
    "admin",
    "Admin@123",
    "invalid-code"
)

Layer 1: Checking capability code...
Capability verification failed.
Access denied.


In [22]:
print("========== SECURITY SUMMARY ==========")
print("✓ AES-256 encryption implemented")
print("✓ Password hashing implemented")
print("✓ Parameterized SQL queries implemented")
print("✓ Suspicious SQL input rejected")
print("✓ Capability-code verification implemented")
print("✓ Double-layer access control implemented")
print("✓ Sensitive information protected")
print("======================================")

========== SECURITY SUMMARY ==========
✓ AES-256 encryption implemented
✓ Password hashing implemented
✓ Parameterized SQL queries implemented
✓ Suspicious SQL input rejected
✓ Capability-code verification implemented
✓ Double-layer access control implemented
✓ Sensitive information protected


In [23]:
connection.close()

print("Database connection closed successfully.")

Database connection closed successfully.
